# 05 Roofs: generate, detect, cut, measure depth, name the type

Four roof photos are generated on the GPU (SDXL-Turbo, 1 step each), then the same stack as notebook 02 runs on them: Grounding DINO (roof, dormer, window, chimney), SAM 2.1 (cut the roof out), Depth Anything V2, CLIP (roof type). Times are measured per stage. The images are generated, not photographs: say so wherever they are shown. Check the SDXL-Turbo licence before using the pictures beyond research.

In [ ]:
import json, time, numpy as np, torch
from pathlib import Path
from PIL import Image, ImageDraw
OUT = Path('/workspace/usable-space/out/roofs'); OUT.mkdir(parents=True, exist_ok=True)
dev = 'cuda'
T = {}
def tick(name, t0): torch.cuda.synchronize(); T[name] = round(time.time() - t0, 2); print(name, T[name], 's')
print(torch.cuda.get_device_name(0), f'{torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB free')

In [ ]:
from diffusers import AutoPipelineForText2Image
PROMPTS = {
  'gable': 'a photograph of a German family house with a gable roof of red clay tiles and two dormer windows, blue sky, sharp, realistic',
  'hip': 'a photograph of a German house with a hip roof of dark grey tiles and a chimney, clear day, sharp, realistic',
  'mansard': 'a photograph of a house with a mansard roof and dormer windows, slate roof, overcast sky, sharp, realistic',
  'flat': 'a photograph of a modern white house with a flat roof, large windows, blue sky, sharp, realistic'}
t0 = time.time()
pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16').to(dev)
tick('load_sdxl_turbo', t0)
imgs = {}
t0 = time.time()
for k, p in PROMPTS.items():
    g = torch.Generator(dev).manual_seed(11)
    imgs[k] = pipe(p, num_inference_steps=1, guidance_scale=0.0, height=512, width=512, generator=g).images[0]
tick('generate_4_roofs', t0)
T['peak_gpu_gib_generate'] = round(torch.cuda.max_memory_allocated() / 2**30, 1)
del pipe; torch.cuda.empty_cache()
for k, im in imgs.items(): im.save(OUT / f'roof_{k}.jpg', quality=92)

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
gp = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-tiny')
gm = AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-tiny').to(dev).eval()
PROMPT = 'a roof. a dormer. a window. a chimney.'
dets = {}
t0 = time.time()
for k, im in imgs.items():
    inp = gp(images=im, text=PROMPT, return_tensors='pt').to(dev)
    with torch.no_grad(): out = gm(**inp)
    kw = dict(threshold=0.25, text_threshold=0.2, target_sizes=[im.size[::-1]])
    try: r = gp.post_process_grounded_object_detection(out, inp.input_ids, **kw)[0]
    except TypeError:
        kw['box_threshold'] = kw.pop('threshold'); r = gp.post_process_grounded_object_detection(out, inp.input_ids, **kw)[0]
    labels = r.get('text_labels', r.get('labels'))
    dets[k] = [{'label': str(l), 'score': round(float(s), 2), 'box': [round(float(v)) for v in b]} for l, s, b in zip(labels, r['scores'], r['boxes'])]
    print(k, [(d['label'], d['score']) for d in dets[k]])
tick('grounding_dino_4', t0)
del gm; torch.cuda.empty_cache()

In [ ]:
from transformers import Sam2Processor, Sam2Model
sp = Sam2Processor.from_pretrained('facebook/sam2.1-hiera-tiny')
sm = Sam2Model.from_pretrained('facebook/sam2.1-hiera-tiny').to(dev).eval()
masks = {}
t0 = time.time()
for k, im in imgs.items():
    ds = dets[k]
    roofs = [d for d in ds if 'roof' in d['label']] or ds[:1]
    if not roofs: masks[k] = None; continue
    box = max(roofs, key=lambda d: d['score'])['box']
    inp = sp(images=im, input_boxes=[[box]], return_tensors='pt').to(dev)
    with torch.no_grad(): out = sm(**inp, multimask_output=False)
    masks[k] = sp.post_process_masks(out.pred_masks.cpu(), inp['original_sizes'])[0][0, 0].numpy() > 0
    print(k, 'roof mask', round(100 * masks[k].mean(), 1), '% of the picture')
tick('sam2_4', t0)
del sm; torch.cuda.empty_cache()

In [ ]:
from transformers import pipeline, CLIPModel, CLIPProcessor
t0 = time.time()
dp = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf', device=0)
depths = {k: np.array(dp(im)['depth']) for k, im in imgs.items()}
tick('depth_4', t0)
del dp; torch.cuda.empty_cache()
cm = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(dev).eval()
cp = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
TYPES = ['a gable roof', 'a hip roof', 'a mansard roof', 'a flat roof', 'a shed roof']
key = {'gable': 0, 'hip': 1, 'mansard': 2, 'flat': 3}
clip = {}
t0 = time.time()
for k, im in imgs.items():
    inp = cp(text=TYPES, images=im, return_tensors='pt', padding=True).to(dev)
    with torch.no_grad(): lg = cm(**inp).logits_per_image[0]
    p = torch.softmax(lg, dim=0).cpu().numpy()
    clip[k] = {t.replace('a ', ''): round(float(v), 3) for t, v in zip(TYPES, p)}
    print(k, 'asked for', k, '-> CLIP says', TYPES[int(p.argmax())], round(float(p.max()), 2))
tick('clip_4', t0)
del cm; torch.cuda.empty_cache()

In [ ]:
W = 340
def fit(im): return im.resize((W, W))
def overlay(im, ds, m):
    base = np.array(im).astype(float)
    if m is not None: base[m] = base[m] * 0.4 + np.array([232, 69, 44]) * 0.6
    o = Image.fromarray(base.astype('uint8')); d = ImageDraw.Draw(o)
    for x in ds:
        c = (232, 69, 44) if 'roof' in x['label'] else (255, 255, 255)
        d.rectangle(x['box'], outline=c, width=3); d.text((x['box'][0] + 4, x['box'][1] + 2), f"{x['label']} {x['score']}", fill=(255, 255, 255))
    return o
def dmap(dep):
    v = (dep - dep.min()) / (dep.max() - dep.min() + 1e-6); return Image.fromarray((255 * (1 - v)).astype('uint8')).convert('RGB')
keys = list(imgs)
sheet = Image.new('RGB', (W * 3, W * len(keys)), (255, 255, 255))
for r, k in enumerate(keys):
    for c, t in enumerate([fit(imgs[k]), fit(overlay(imgs[k], dets[k], masks[k])), fit(dmap(depths[k]).resize(imgs[k].size))]): sheet.paste(t, (c * W, r * W))
sheet.save(OUT / 'roofs_montage.jpg', quality=86)
T['gpu'] = torch.cuda.get_device_name(0)
(OUT / 'roofs_results.json').write_text(json.dumps({'prompts': PROMPTS, 'detections': dets, 'clip': clip, 'timings_s': T, 'roof_mask_pct': {k: (round(100 * float(m.mean()), 1) if m is not None else None) for k, m in masks.items()}}, indent=1), encoding='utf-8')
print(json.dumps(T, indent=1)); print((OUT / 'roofs_montage.jpg').stat().st_size // 1024, 'KB')